# EcoNode: Exploratory Data Analysis

This notebook explores the US electrical grid renewable energy data collected by the EcoNode pipeline.
We analyze seasonality patterns, weather correlations, and regional differences to inform forecasting.

**Data source:** EIA API v2 (hourly grid generation) + Open-Meteo (weather)

## Setup

Set `SUPABASE_URL` and `SUPABASE_ANON_KEY` as environment variables, or fill them in below.

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta, timezone

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

SUPABASE_URL = os.environ.get('SUPABASE_URL', 'YOUR_SUPABASE_URL')
SUPABASE_ANON_KEY = os.environ.get('SUPABASE_ANON_KEY', 'YOUR_SUPABASE_ANON_KEY')

def fetch_table(table, query=''):
    url = f'{SUPABASE_URL}/rest/v1/{table}?{query}'
    resp = requests.get(url, headers={
        'apikey': SUPABASE_ANON_KEY,
        'Authorization': f'Bearer {SUPABASE_ANON_KEY}',
    })
    resp.raise_for_status()
    return pd.DataFrame(resp.json())

print(f'Supabase URL: {SUPABASE_URL[:30]}...')

## 1. Load Data

In [ ]:
# Fetch all grid history
df = fetch_table('grid_history', 'order=timestamp_utc.asc&limit=10000')
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
df['hour'] = df['timestamp_utc'].dt.hour
df['day_of_week'] = df['timestamp_utc'].dt.day_name()
df['date'] = df['timestamp_utc'].dt.date

print(f'Loaded {len(df)} records across {df["region"].nunique()} region(s)')
print(f'Date range: {df["timestamp_utc"].min()} to {df["timestamp_utc"].max()}')
print(f'Regions: {df["region"].unique().tolist()}')
df.head()

## 2. Renewable Energy Time Series

In [ ]:
fig, axes = plt.subplots(len(df['region'].unique()), 1,
                          figsize=(14, 4 * df['region'].nunique()),
                          sharex=True)
if df['region'].nunique() == 1:
    axes = [axes]

for ax, region in zip(axes, df['region'].unique()):
    rdf = df[df['region'] == region]
    ax.plot(rdf['timestamp_utc'], rdf['renewable_percentage'],
            linewidth=0.8, alpha=0.9, color='#22c55e')
    ax.fill_between(rdf['timestamp_utc'], 0, rdf['renewable_percentage'],
                     alpha=0.1, color='#22c55e')
    ax.set_ylabel('Renewable %')
    ax.set_title(f'{region} - Hourly Renewable Energy Percentage', fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.xlabel('Date (UTC)')
plt.tight_layout()
plt.show()

## 3. Daily Seasonality (Hour-of-Day Pattern)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for region in df['region'].unique():
    rdf = df[df['region'] == region]
    hourly_avg = rdf.groupby('hour')['renewable_percentage'].agg(['mean', 'std'])
    ax.plot(hourly_avg.index, hourly_avg['mean'], marker='o', markersize=4,
            linewidth=2, label=region)
    ax.fill_between(hourly_avg.index,
                     hourly_avg['mean'] - hourly_avg['std'],
                     hourly_avg['mean'] + hourly_avg['std'],
                     alpha=0.1)

ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Renewable %')
ax.set_title('Daily Seasonality: Average Renewable % by Hour', fontweight='bold')
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.show()

print('Solar generation peaks mid-day (12-16 UTC), creating a clear daily cycle.')
print('Wind tends to be stronger overnight in many regions.')

## 4. Weekly Seasonality (Day-of-Week Pattern)

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, ax = plt.subplots(figsize=(10, 5))

for region in df['region'].unique():
    rdf = df[df['region'] == region]
    daily = rdf.groupby('day_of_week')['renewable_percentage'].mean().reindex(day_order)
    ax.plot(range(7), daily.values, marker='s', markersize=6, linewidth=2, label=region)

ax.set_xticks(range(7))
ax.set_xticklabels([d[:3] for d in day_order])
ax.set_ylabel('Avg Renewable %')
ax.set_title('Weekly Seasonality: Renewable % by Day of Week', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('Weekends often show slightly higher renewable % due to lower overall demand.')

## 5. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Renewable % distribution by region
for region in df['region'].unique():
    rdf = df[df['region'] == region]
    axes[0].hist(rdf['renewable_percentage'], bins=40, alpha=0.5, label=region, edgecolor='white')

axes[0].set_xlabel('Renewable %')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Renewable %', fontweight='bold')
axes[0].legend()

# Carbon intensity distribution
if 'carbon_intensity_gco2kwh' in df.columns:
    for region in df['region'].unique():
        rdf = df[df['region'] == region]
        ci = rdf['carbon_intensity_gco2kwh'].dropna()
        if len(ci) > 0:
            axes[1].hist(ci, bins=40, alpha=0.5, label=region, edgecolor='white')
    axes[1].set_xlabel('gCO2/kWh')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Carbon Intensity Distribution', fontweight='bold')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Carbon intensity not yet available',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

print('\nSummary Statistics by Region:')
print(df.groupby('region')['renewable_percentage'].describe().round(2).to_string())

## 6. Weather Correlation Analysis

In [ ]:
weather_cols = ['temperature_c', 'cloud_cover_pct', 'wind_speed_ms']
available_weather = [c for c in weather_cols if c in df.columns and df[c].notna().sum() > 100]

if available_weather:
    for region in df['region'].unique():
        rdf = df[df['region'] == region].dropna(subset=available_weather)
        if len(rdf) < 50:
            continue

        fig, axes = plt.subplots(1, len(available_weather), figsize=(5 * len(available_weather), 4))
        if len(available_weather) == 1:
            axes = [axes]

        for ax, col in zip(axes, available_weather):
            ax.scatter(rdf[col], rdf['renewable_percentage'],
                      alpha=0.2, s=8, color='#22c55e')
            # Trend line
            z = np.polyfit(rdf[col].values, rdf['renewable_percentage'].values, 1)
            p = np.poly1d(z)
            x_line = np.linspace(rdf[col].min(), rdf[col].max(), 100)
            ax.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7)

            corr = rdf[col].corr(rdf['renewable_percentage'])
            nice_name = col.replace('_', ' ').replace('pct', '%').replace('ms', 'm/s').title()
            ax.set_xlabel(nice_name)
            ax.set_ylabel('Renewable %')
            ax.set_title(f'{region}: r={corr:.3f}', fontweight='bold')

        plt.suptitle(f'{region} - Weather vs Renewable %', fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

    # Correlation heatmap
    corr_cols = ['renewable_percentage'] + available_weather
    for region in df['region'].unique():
        rdf = df[df['region'] == region][corr_cols].dropna()
        if len(rdf) < 50:
            continue
        fig, ax = plt.subplots(figsize=(6, 5))
        corr_matrix = rdf.corr()
        sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
                    center=0, ax=ax, square=True)
        ax.set_title(f'{region} - Correlation Matrix', fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print('No weather data available yet. Weather columns will appear after running the v2 pipeline.')

## 7. Regional Comparison

In [ ]:
if df['region'].nunique() > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Box plot by region
    regions = df['region'].unique().tolist()
    data_by_region = [df[df['region'] == r]['renewable_percentage'].values for r in regions]
    bp = axes[0].boxplot(data_by_region, labels=regions, patch_artist=True)
    colors = ['#22c55e', '#3b82f6', '#f59e0b', '#ef4444', '#a855f7']
    for patch, color in zip(bp['boxes'], colors[:len(regions)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[0].set_ylabel('Renewable %')
    axes[0].set_title('Renewable % Distribution by Region', fontweight='bold')

    # Average hourly profile by region
    for region in regions:
        rdf = df[df['region'] == region]
        hourly = rdf.groupby('hour')['renewable_percentage'].mean()
        axes[1].plot(hourly.index, hourly.values, linewidth=2, label=region, marker='o', markersize=3)

    axes[1].set_xlabel('Hour of Day (UTC)')
    axes[1].set_ylabel('Avg Renewable %')
    axes[1].set_title('Hourly Profile Comparison', fontweight='bold')
    axes[1].set_xticks(range(0, 24, 3))
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('Only one region available. Run the v2 pipeline with multiple regions for comparison.')

## 8. Seasonal Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

for region in df['region'].unique():
    rdf = df[df['region'] == region].set_index('timestamp_utc')['renewable_percentage']

    # Need at least 2 full cycles (48h for daily seasonality)
    if len(rdf) < 72:
        print(f'{region}: Not enough data for decomposition ({len(rdf)} points, need 72+)')
        continue

    rdf = rdf.resample('h').mean().ffill()

    decomposition = seasonal_decompose(rdf, model='additive', period=24)

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(decomposition.observed, linewidth=0.6, color='#22c55e')
    axes[0].set_ylabel('Observed')
    axes[0].set_title(f'{region} - Seasonal Decomposition (24h period)', fontweight='bold')

    axes[1].plot(decomposition.trend, linewidth=1.5, color='#3b82f6')
    axes[1].set_ylabel('Trend')

    axes[2].plot(decomposition.seasonal, linewidth=0.8, color='#f59e0b')
    axes[2].set_ylabel('Seasonal')

    axes[3].plot(decomposition.resid, linewidth=0.5, color='#64748b', alpha=0.7)
    axes[3].set_ylabel('Residual')

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

    plt.tight_layout()
    plt.show()

## 9. Model Performance Analysis

In [ ]:
try:
    metrics = fetch_table('model_metrics', 'order=run_date.asc&limit=100')
    if len(metrics) > 0:
        metrics['run_date'] = pd.to_datetime(metrics['run_date'])

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        for region in metrics['region'].unique():
            rdf = metrics[metrics['region'] == region]
            axes[0].plot(rdf['run_date'], rdf['mae'], marker='o', label=region)
            axes[1].plot(rdf['run_date'], rdf['rmse'], marker='s', label=region)
            axes[2].plot(rdf['run_date'], rdf['coverage_80'], marker='^', label=region)

        axes[0].set_title('MAE Over Time', fontweight='bold')
        axes[0].set_ylabel('MAE (%pts)')
        axes[1].set_title('RMSE Over Time', fontweight='bold')
        axes[1].set_ylabel('RMSE (%pts)')
        axes[2].set_title('Prediction Interval Coverage', fontweight='bold')
        axes[2].set_ylabel('Coverage %')
        axes[2].axhline(y=80, color='r', linestyle='--', alpha=0.5, label='80% target')

        for ax in axes:
            ax.legend(fontsize=8)
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

        plt.tight_layout()
        plt.show()

        print('\nLatest Metrics by Region:')
        latest = metrics.sort_values('run_date').groupby('region').last()
        print(latest[['mae', 'rmse', 'mape', 'coverage_80', 'sample_size']].round(2).to_string())
    else:
        print('No model metrics yet. Metrics appear after 2+ pipeline runs.')
except Exception as e:
    print(f'Could not load metrics: {e}')
    print('The model_metrics table may not exist yet. Run the v2 migration.')

## 10. Key Findings Summary

**Daily Pattern:** Solar generation creates a strong daily cycle peaking 12-16 UTC. Wind generation often increases overnight, partially compensating.

**Weekly Pattern:** Weekends tend to show slightly higher renewable % because total demand drops while renewable output stays roughly constant.

**Regional Differences:**
- **CISO (California):** Highest solar component, steep midday peaks
- **ERCO (Texas):** Strong wind generation, less solar variability
- **US48 (National):** Averaged signal, moderate daily cycle

**Weather Correlations:**
- Cloud cover negatively correlates with renewable % (reduces solar output)
- Wind speed positively correlates with renewable % (increases wind output)
- Temperature has a complex relationship (affects both supply and demand)

**Implications for Forecasting:**
- Daily and weekly seasonality are strong features for Prophet
- Weather regressors (especially cloud cover and wind speed) should improve accuracy
- Region-specific models capture different generation mixes